In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mitygacja błędów za pomocą sieci tensorowych (TEM): Funkcja Qiskit firmy Algorithmiq
> **Note:** Funkcje Qiskit to eksperymentalna funkcjonalność dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan i On-Prem (przez IBM Quantum Platform API). Są w fazie podglądu przedpremierowego i mogą ulec zmianie.


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Przegląd
Metoda Mitygacji Błędów za pomocą Sieci Tensorowych (TEM) firmy Algorithmiq to hybrydowy
algorytm kwantowo-klasyczny zaprojektowany do przeprowadzania mitygacji szumów całkowicie
na etapie klasycznego post-processingu. Dzięki TEM możesz obliczać
wartości oczekiwane obserwowalnych, mitygując nieuniknione błędy wywoływane przez szumy,
które pojawiają się na sprzęcie kwantowym — z większą dokładnością i efektywnością kosztową,
co czyni tę metodę bardzo atrakcyjną zarówno dla badaczy kwantowych, jak i praktyków z przemysłu.

Metoda polega na skonstruowaniu sieci tensorowej reprezentującej odwrotność
globalnego kanału szumowego wpływającego na stan procesora kwantowego,
a następnie zastosowaniu tego odwzorowania do informacyjnie pełnych wyników pomiarów
uzyskanych z zaszumionego stanu w celu uzyskania nieobciążonych estymatorów obserwowalnych.

Zaletą TEM jest to, że wykorzystuje informacyjnie pełne pomiary, dając dostęp
do szerokiego zestawu mitygowanych wartości oczekiwanych obserwowalnych i osiągając
optymalny narzut próbkowania na sprzęcie kwantowym, zgodnie z opisem w Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), oraz Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Narzut
pomiarowy odnosi się do liczby dodatkowych pomiarów wymaganych do
przeprowadzenia efektywnej mitygacji błędów — jest to kluczowy czynnik dla wykonalności
obliczeń kwantowych. Dlatego TEM ma potencjał, by umożliwić przewagę kwantową
w złożonych scenariuszach, takich jak zastosowania w dziedzinie chaosu kwantowego,
fizyki wielu ciał, dynamiki Hubbarda oraz symulacji chemii małych cząsteczek.

Główne cechy i korzyści TEM można podsumować następująco:

1.  **Optymalny narzut pomiarowy**: TEM jest optymalna względem
[granic teoretycznych](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
co oznacza, że żadna metoda nie może osiągnąć mniejszego narzutu pomiarowego. Innymi
słowy, TEM wymaga minimalnej liczby dodatkowych pomiarów do przeprowadzenia
mitygacji błędów. Przekłada się to bezpośrednio na minimalne zużycie czasu kwantowego.
2.  **Efektywność kosztowa**: Ponieważ TEM obsługuje mitygację szumów całkowicie w etapie
post-processingu, nie ma potrzeby dodawania dodatkowych obwodów do komputera kwantowego,
co nie tylko obniża koszty obliczeń, ale także zmniejsza ryzyko wprowadzenia dodatkowych
błędów wynikających z niedoskonałości urządzeń kwantowych.
3.  **Estymacja wielu obserwowalnych**: Dzięki informacyjnie pełnym pomiarom
TEM efektywnie estymuje wiele obserwowalnych na podstawie tych samych danych pomiarowych
z komputera kwantowego.
4.  **Mitygacja błędów odczytu**: Funkcja Qiskit TEM zawiera również
[zastrzeżoną metodę mitygacji błędów odczytu](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
zdolną do znacznego zmniejszenia błędów readout po krótkim przebiegu kalibracyjnym.
5.  **Dokładność**: TEM znacząco poprawia dokładność i niezawodność
cyfrowych symulacji kwantowych, czyniąc algorytmy kwantowe bardziej precyzyjnymi
i wiarygodnymi.
## Opis
Funkcja TEM pozwala uzyskać mitygowane błędy wartości oczekiwane dla
wielu obserwowalnych w obwodzie kwantowym przy minimalnym narzucie próbkowania. Circuit
jest mierzony za pomocą informacyjnie pełnej miary z operatorem o wartości dodatniej (IC-POVM),
a zebrane wyniki pomiarów są przetwarzane na komputerze klasycznym. Ten pomiar służy do
wykonywania metod sieci tensorowych i zbudowania mapy odwrotności szumu. Funkcja stosuje
odwzorowanie, które w pełni odwraca cały zaszumiony Circuit za pomocą sieci tensorowych
reprezentujących zaszumione warstwy.

![Schemat TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Mitygowana błędem estymacja obserwowanej wartości O poprzez post-processing wyników pomiarów zaszumionego procesora kwantowego. U i N oznaczają idealną operację kwantową i powiązaną mapę szumu, która może być ogólnie nielokalna (i rozszerzona do szarych ramek). D oznacza tensor operatorów dualnych do efektów w pomiarze IC. Moduł mitygacji szumu M jest siecią tensorową efektywnie kontrahowaną od środka na zewnątrz. Pierwsza iteracja kontrakcji jest przedstawiona linią kropkowaną w kolorze fioletowym, druga linią przerywaną, a trzecia linią ciągłą.")

Po przesłaniu obwodów do funkcji są one transpilowane i
optymalizowane w celu zminimalizowania liczby warstw z bramkami dwu-qubitowymi (głośniejszymi
bramkami na urządzeniach kwantowych). Szum wpływający na warstwy jest uczony przy użyciu
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
z zastosowaniem rzadkiego modelu szumu Pauli-Lindblad, zgodnie z opisem w E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model szumu jest dokładnym opisem szumu na urządzeniu, zdolnym do
uchwycenia subtelnych cech, w tym przesłuchów między qubitami. Jednak szum na
urządzeniach może fluktuować i dryfować, a nauczony szum może nie być dokładny
w momencie przeprowadzania estymacji. Może to prowadzić do niedokładnych wyników.
## Pierwsze kroki
Uwierzytelnij się za pomocą [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz funkcję TEM w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w środowisku lokalnym.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Przykład
Poniższy fragment kodu pokazuje przykład, w którym TEM jest używana do obliczenia wartości oczekiwanych obserwowanej wartości dla prostego Circuit kwantowego.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Użyj interfejsów API Qiskit Serverless, aby sprawdzić status zadania funkcji Qiskit:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Możesz pobrać wyniki w następujący sposób:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Oczekiwana wartość dla Circuit bez szumu dla podanego operatora powinna wynosić około `0.18409094298943401`.
## Dane wejściowe
**Parametry**

Nazwa | Typ | Opis | Wymagane | Domyślna | Przykład
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Iterowalny zbiór obiektów podobnych do PUB (primitive unified bloc), takich jak krotki `(circuit, observables)` lub `(circuit, observables, parameters, precision)`. Zobacz [Przegląd PUB](/guides/primitive-input-output#overview-of-pubs), aby uzyskać więcej informacji. Jeśli zostanie przekazany Circuit nie-ISA, zostanie on transpilowany z optymalnymi ustawieniami. Jeśli zostanie przekazany Circuit ISA, nie będzie transpilowany; w tym przypadku obserwowalna musi być zdefiniowana na całym QPU. | Tak | N/A | (circuit, observables)
`backend_name` | str | Nazwa Backend, do którego kierowane jest zapytanie.| Nie | Jeśli nie podano, zostanie użyty najmniej obciążony Backend. | "ibm_fez"
`options` | dict | Opcje wejściowe. Więcej szczegółów w sekcji `Options`. | Nie | Szczegóły w sekcji `Options`.| {"max_bond_dimension": 100\}

> **Caution:** TEM ma obecnie następujące ograniczenia:
> 
>   - Parametryzowane Circuit nie są obsługiwane. Argument parametrów powinien być ustawiony na `None`, jeśli podano precyzję. To ograniczenie zostanie usunięte w przyszłych wersjach.
>   - Obsługiwane są tylko Circuit bez pętli. To ograniczenie zostanie usunięte w przyszłych wersjach.
>   - Bramki nie-unitarne, takie jak reset, measure oraz wszelkie formy przepływu sterowania nie są obsługiwane. Obsługa resetu zostanie dodana w nadchodzących wydaniach.
### Opcje
Słownik zawierający zaawansowane opcje dla TEM. Słownik może zawierać klucze z poniższej tabeli. Jeśli którakolwiek z opcji nie zostanie podana, zostanie użyta wartość domyślna wymieniona w tabeli. Wartości domyślne są odpowiednie do typowego użycia TEM.

Nazwa | Wartości | Opis | Domyślna
-- | -- | -- | --
`tem_max_bond_dimension` | int | Maksymalny wymiar więzi do użycia w sieciach tensorowych. | 500 |
`tem_compression_cutoff` | float | Wartość odcięcia dla sieci tensorowych. | 1e-16
`compute_shadows_bias_from_observable` | bool | Flaga logiczna wskazująca, czy odchylenie dla protokołu pomiarowego klasycznych cieni powinno być dostosowane do obserwowalnej PUB. Jeśli False, zostanie użyty standardowy protokół klasycznych cieni (równe prawdopodobieństwo pomiaru Z, X, Y).| False
`shadows_bias` | np.ndarray | Odchylenie do użycia w protokole losowych klasycznych cieni — jednowymiarowa lub dwuwymiarowa tablica o rozmiarze 3 lub kształcie (num_qubits, 3). Kolejność: ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int lub `None` | Maksymalny czas wykonania na QPU w sekundach. Jeśli czas wykonania przekroczy tę wartość, zadanie zostanie anulowane. Jeśli `None`, zostanie zastosowany domyślny limit ustawiony przez Qiskit Runtime. | `None`
`num_randomizations` | int | Liczba randomizacji do użycia przy uczeniu szumu i twirling bramek. | 32
`max_layers_to_learn` | int | Maksymalna liczba unikalnych warstw do nauczenia. | 4
`mitigate_readout_error` | bool | Flaga logiczna wskazująca, czy należy przeprowadzić mitygację błędów odczytu. | True
`num_readout_calibration_shots` | int | Liczba strzałów do użycia przy mitygacji błędów odczytu. | 10000
`default_precision` | float | Domyślna precyzja dla PUB, dla których precyzja nie jest określona. |0.02
`seed` | int lub `None` | Ustawia ziarno generatora liczb losowych dla odtwarzalności. Jeśli `None`, ziarno nie jest ustawiane. | None
## Dane wyjściowe
Obiekt Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) zawierający wynik mitygowany przez TEM. Wynik dla każdego PUB jest zwracany jako [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) zawierający następujące pola:

Nazwa | Typ | Opis
-- | -- | --
data | DataBin | Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) zawierający mitygowaną przez TEM obserwowalną i jej błąd standardowy. DataBin ma następujące pola: <ul><li>`evs`: Wartość obserwowanej mitygowanej przez TEM.</li><li>`stds`: Błąd standardowy obserwowanej mitygowanej przez TEM.</li></ul>
metadata | dict | Słownik zawierający dodatkowe wyniki. Słownik zawiera następujące klucze: <ul><li>`"evs_non_mitigated"`: Wartość obserwowanej bez mitygacji błędów.</li><li>`"stds_non_mitigated"`: Błąd standardowy wyniku bez mitygacji błędów.</li><li>`"evs_mitigated_no_readout_mitigation"`: Wartość obserwowanej z mitygacją błędów, ale bez mitygacji błędów odczytu.</li><li>`"stds_mitigated_no_readout_mitigation"`: Błąd standardowy wyniku z mitygacją błędów, ale bez mitygacji błędów odczytu.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Wartość obserwowanej bez mitygacji błędów, ale z mitygacją błędów odczytu.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Błąd standardowy wyniku bez mitygacji błędów, ale z mitygacją błędów odczytu.</li></ul>
## Pobieranie komunikatów o błędach
Jeśli status zadania to ERROR, użyj `job.result()`, aby pobrać komunikat o błędzie w następujący sposób: